**Imports**

In [0]:
import pyspark.sql.functions as F

**Email validation**

In [0]:
customer_df=spark.read.table("rental.bike_rental_bronze.customer")
email_regex = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

# 3. Add a boolean flag column indicating validity status
# We handle Null values safely using F.coalesce or F.when
customer_mail_validated_df = customer_df.withColumn(
    "is_valid_email",
    F.when(F.col("email").isNull(), False)
     .otherwise(F.col("email").rlike(email_regex))
)

In [0]:
invalid_emails_df = customer_mail_validated_df.filter(F.col("is_valid_email") == False)
if invalid_emails_df.count() > 0:
    Exception ("Invalid emails found")

**DeDuplicates**

In [0]:
customer_mail_validated_df = customer_mail_validated_df.dropDuplicates()

**Names Standardlization**

In [0]:
customer_mail_validated_df = customer_mail_validated_df.withColumn(
    "name",
    F.initcap(
        F.regexp_replace(
            F.trim(F.lower(F.col("name"))),
            r"\s+",
            " "
        )
    )
)

**Customer Status**

In [0]:
customer_mail_validated_df = customer_mail_validated_df.withColumn("status", F.lit("Active"))

In [0]:
customer_mail_validated_df.write.mode("overwrite").saveAsTable("rental.bike_rental_silver.customer")